In [3]:
import pyspark
from pyspark.sql import SparkSession
import os                                            
os.environ['HADOOP_HOME'] = 'C:\\Program Files\\hadoop\\hadoop-3.3.6'
os.environ['PATH'] += ';C:\\Program Files\\hadoop\\hadoop-3.3.6\\bin'

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('teste') \
    .getOrCreate()

In [5]:
!curl -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 67.8M    0  237k    0     0   154k      0  0:07:29  0:00:01  0:07:28  154k
  1 67.8M    1  979k    0     0   386k      0  0:02:59  0:00:02  0:02:57  386k
  2 67.8M    2 1459k    0     0   411k      0  0:02:48  0:00:03  0:02:45  411k
  3 67.8M    3 2195k    0     0   483k      0  0:02:23  0:00:04  0:02:19  483k
  3 67.8M    3 2771k    0     0   500k      0  0:02:18  0:00:05  0:02:13  588k
  4 67.8M    4 3379k    0     0   516k      0  0:02:14  0:00:06  0:02:08  628k
  5 67.8M    5 3971k    0     0   526k      0  0:02:11  0:00:07  0:02:04  597k
  6 67.8M    6 4835k    0     0   564k      0  0:02:02  0:00:08  0:01:54  673k
  7 67.8M    7 5443k    0     0   570k      0  0:02

1° Question - Answer 

In [ ]:
spark.version

'4.1.1'

2° Question - Answer 

In [14]:
folder = 'output/yellow_taxi/'
files = [f for f in os.listdir(folder) if f.endswith('.parquet')]
sizes = [os.path.getsize(os.path.join(folder, f)) for f in files]
print(sizes)
print(sum(sizes) / len(sizes) / 1024 / 1024, 'MB')

[25577435, 25608464, 25602315, 25594240]
24.409879207611084 MB


In [15]:
df_yellow = spark.read.parquet('output/yellow_taxi/')

In [16]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [ ]:
df_yellow.select('VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime') \
    .filter("VendorID == 2") \
    .show(5)

+--------+--------------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------+--------------------+---------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|
|       2| 2025-11-06 14:01:48|  2025-11-06 14:25:53|
|       2| 2025-11-07 16:53:08|  2025-11-07 17:10:10|
|       2| 2025-11-09 10:55:05|  2025-11-09 10:58:57|
|       2| 2025-11-03 13:35:09|  2025-11-03 13:47:48|
+--------+--------------------+---------------------+
only showing top 5 rows


In [17]:
df_yellow.createOrReplaceTempView('yellow_taxi')

In [26]:
spark.sql("""
SELECT * FROM yellow_taxi LIMIT 5;           
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

3° Question - Answer

In [32]:
spark.sql("""
SELECT
    COUNT(*) 
FROM yellow_taxi 
WHERE tpep_pickup_datetime BETWEEN '2025-11-15' AND '2025-11-16';
""").show()

+--------+
|count(1)|
+--------+
|  162608|
+--------+



4° Question - Answer

In [38]:
from pyspark.sql.functions import col, unix_timestamp

df = df.withColumn(
    'trip_duration',
    (unix_timestamp(col('tpep_dropoff_datetime')) - 
     unix_timestamp(col('tpep_pickup_datetime'))) / 3600
)

df.createOrReplaceTempView('yellow_taxi')

In [43]:
spark.sql("""

SELECT trip_duration
FROM yellow_taxi
order by trip_duration desc
LIMIT 1

""").show()

+-----------------+
|    trip_duration|
+-----------------+
|90.64666666666666|
+-----------------+



5° Question - Answer

4040

In [2]:
!curl -O https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  12331 100  12331   0      0  82273      0                              0
100  12331 100  12331   0      0  82158      0                              0
100  12331 100  12331   0      0  81994      0                              0


In [10]:
df_zone_lookup = spark.read.csv('taxi_zone_lookup.csv', header=True)
df_zone_lookup.createOrReplaceTempView('taxi_zone_lookup')

In [11]:
df_zone_lookup.show(2)

+----------+-------+--------------+------------+
|LocationID|Borough|          Zone|service_zone|
+----------+-------+--------------+------------+
|         1|    EWR|Newark Airport|         EWR|
|         2| Queens|   Jamaica Bay|   Boro Zone|
+----------+-------+--------------+------------+
only showing top 2 rows


In [20]:
yellow_with_lookup = df_yellow.join(df_zone_lookup, df_yellow.PULocationID == df_zone_lookup.LocationID)

In [21]:
yellow_with_lookup.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

6° Question - Answer

In [39]:
from pyspark.sql.functions import count, col

yellow_with_lookup.filter(col('Zone').isin(
    "Governor's Island/Ellis Island/Liberty Island",
    "Arden Heights",
    "Rikers Island",
    "Jamaica Bay"
)).groupBy('Zone') \
      .agg(count('*').alias('total_trips')) \
      .orderBy('total_trips') \
      .first()

Row(Zone="Governor's Island/Ellis Island/Liberty Island", total_trips=1)